# Load Dataset

In [ ]:
import json
import pandas as pd

path = "/kaggle/input/dataaaa/code_features.jsonl"

records = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)

df.head()


In [ ]:
df.info()

# Prepare Features & Labels

### Feature Importance Summary

**Highly Important**
- `num_loops` — Direct indicator of iteration count.
- `max_loop_depth` — Captures polynomial growth (e.g., nested loops).
- `has_nested_loops` — Distinguishes linear vs quadratic behavior.
- `has_log_update` — Identifies logarithmic patterns (e.g., halving).
- `uses_sort` — Sorting implies O(n log n).
- `recursion_flag` — Detects recursive algorithms.
- `num_recursive_calls` — Differentiates linear vs exponential recursion.
- `uses_comprehension` — Reveals hidden linear iteration.

**Moderately Important**
- `num_for`, `num_while` — Provide loop context.
- `has_break`, `has_continue` — Control-flow context only.
- `has_early_return` — Affects best-case, not worst-case.
- `uses_generator` — Often linear, sometimes lazy.
- `num_function_calls` — Helpful only with other features.

**Low / Not Important**
- `uses_list`, `uses_dict`, `uses_set`, `uses_tuple` — Common, not discriminative.
- `num_return` — Mostly stylistic.
- `representation` — Constant across samples.


In [ ]:
DROP_COLUMNS = [
    "code",
    "representation",
    "uses_list",
    "uses_dict",
    "uses_set",
    "uses_tuple",
    "num_return",
    "num_function_calls"
]

df = df.drop(columns=DROP_COLUMNS)
df.head()

### Shuffle Data

The raw dataset may be sorted by class. We shuffle before any processing to avoid ordering bias in splits and batch-based workflows.

In [ ]:
# FIX: Shuffle the full dataframe before splitting to avoid class-ordering bias
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("Dataset shuffled. Shape:", df.shape)

### Splitting

We split **before** encoding to avoid data leakage. `get_dummies` is applied separately on train and test so the model never sees test-set category information.

In [ ]:
from sklearn.model_selection import train_test_split

# FIX: Split on the raw dataframe BEFORE one-hot encoding to prevent leakage
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    stratify=df["complexity"],
    random_state=42
)

print("Train size:", len(df_train))
print("Test size:", len(df_test))

### Encoding

Apply one-hot encoding to train and test independently, then align columns so both have the same feature set.

In [ ]:
# FIX: Apply get_dummies separately on train/test, then reindex test to match train columns
df_train = pd.get_dummies(df_train, columns=["loop_bound_type"], prefix="loop")
df_test  = pd.get_dummies(df_test,  columns=["loop_bound_type"], prefix="loop")

# Align test to train columns (fills any unseen categories with 0)
df_test = df_test.reindex(columns=df_train.columns, fill_value=0)

df_train.head()

In [ ]:
X_train = df_train.drop(columns=["complexity"])
y_train = df_train["complexity"]

X_test  = df_test.drop(columns=["complexity"])
y_test  = df_test["complexity"]

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)

### Sanity Check

In [ ]:
# Ensure no object columns remain in training features
assert X_train.select_dtypes(include="object").empty, "Object columns found in X_train!"

# Ensure no missing values
assert X_train.isnull().sum().sum() == 0, "Missing values found in X_train!"
assert X_test.isnull().sum().sum()  == 0, "Missing values found in X_test!"

# Verify numeric-only dataset
print("X_train dtypes:")
print(X_train.dtypes.value_counts())

# MODEL 1 — Gradient Boosted Trees (XGBoost)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier

In [ ]:
# FIX: Single shared LabelEncoder for both models — avoids risk of divergent class ordering
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

print("Class mapping:", dict(zip(le.classes_, range(len(le.classes_)))))

In [ ]:
# FIX: Set n_jobs=1 on the estimator when RandomizedSearchCV already parallelises with n_jobs=-1
# to avoid thread over-subscription (n_cpu x n_cpu threads)
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(le.classes_),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=1   # <-- was -1; set to 1 because the search itself uses n_jobs=-1
)

In [ ]:
xgb_param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [3, 4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3],
    "min_child_weight": [1, 3, 5],
    "reg_alpha": [0, 0.1, 0.5],
    "reg_lambda": [1, 1.5, 2]
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_param_grid,
    n_iter=40,
    scoring="f1_macro",
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

xgb_search.fit(X_train, y_train_enc)

best_xgb = xgb_search.best_estimator_

In [ ]:
from sklearn.metrics import classification_report

y_pred_xgb = best_xgb.predict(X_test)
y_pred_xgb = le.inverse_transform(y_pred_xgb)

print("Best params:", xgb_search.best_params_)
print(classification_report(y_test, y_pred_xgb))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm_xgb = confusion_matrix(y_test, y_pred_xgb, labels=le.classes_)

disp_xgb = ConfusionMatrixDisplay(
    confusion_matrix=cm_xgb,
    display_labels=le.classes_
)

plt.figure(figsize=(8, 6))
disp_xgb.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix — XGBoost (Test Set)")
plt.grid(False)
plt.tight_layout()
plt.show()

# MODEL 2 — Small MLP (Neural Baseline)

In [ ]:
# FIX: Reuse the shared LabelEncoder (le) instead of creating a second one (le_mlp)
# This guarantees both models map classes identically
y_train_mlp = le.transform(y_train)
y_test_mlp  = le.transform(y_test)

print("Class mapping (shared):", dict(zip(le.classes_, range(len(le.classes_)))))

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report

In [ ]:
mlp_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        max_iter=1000,
        early_stopping=True,
        n_iter_no_change=10,
        random_state=42
    ))
])

In [ ]:
mlp_param_grid = {
    "mlp__hidden_layer_sizes": [(32,), (64,), (64, 32)],
    "mlp__activation": ["relu", "tanh"],
    "mlp__alpha": [1e-4, 1e-3, 1e-2],
    "mlp__learning_rate_init": [0.001, 0.005, 0.01]
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

mlp_search = RandomizedSearchCV(
    estimator=mlp_pipeline,
    param_distributions=mlp_param_grid,
    n_iter=25,
    scoring="f1_macro",
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42,
    error_score="raise"
)

mlp_search.fit(X_train, y_train_mlp)

best_mlp = mlp_search.best_estimator_

In [ ]:
y_pred_mlp = best_mlp.predict(X_test)
y_pred_mlp = le.inverse_transform(y_pred_mlp)   # FIX: use shared le, not le_mlp

print("Best parameters:")
print(mlp_search.best_params_)

print(classification_report(y_test, y_pred_mlp))

In [ ]:
mlp_model = best_mlp.named_steps["mlp"]

# FIX: Guard loss_curve_ access — early stopping may produce a short but valid curve
if hasattr(mlp_model, 'loss_curve_') and mlp_model.loss_curve_:
    plt.figure(figsize=(8, 5))
    plt.plot(mlp_model.loss_curve_, label="Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("MLP Training Loss Curve")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("No loss curve available (model may not have converged normally).")

In [ ]:
cm_mlp = confusion_matrix(y_test, y_pred_mlp, labels=le.classes_)

disp_mlp = ConfusionMatrixDisplay(
    confusion_matrix=cm_mlp,
    display_labels=le.classes_
)

plt.figure(figsize=(8, 6))
disp_mlp.plot(cmap="Oranges", xticks_rotation=45)
plt.title("Confusion Matrix — MLP (Test Set)")
plt.grid(False)
plt.tight_layout()
plt.show()

## Save Models

In [ ]:
# Save the 2 trained models so you can download them from Kaggle
# Run this cell after training both models: best_xgb and best_mlp

import joblib
import json
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

OUTPUT_DIR = Path("/kaggle/working/saved_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save XGBoost model + the shared label encoder
joblib.dump(best_xgb, OUTPUT_DIR / "xgboost_model.joblib")
joblib.dump(le,       OUTPUT_DIR / "label_encoder.joblib")   # one encoder for both models

# Save MLP pipeline (already includes StandardScaler + MLP)
joblib.dump(best_mlp, OUTPUT_DIR / "mlp_pipeline.joblib")

# Save feature information needed later by the GUI/inference code
feature_info = {
    "feature_columns": list(X_train.columns),
    "drop_columns": DROP_COLUMNS,
    "target_column": "complexity",
    "categorical_columns": ["loop_bound_type"],
    "notes": "For prediction, create the same features, apply get_dummies, then reindex to feature_columns with fill_value=0."
}

with open(OUTPUT_DIR / "feature_info.json", "w", encoding="utf-8") as f:
    json.dump(feature_info, f, indent=2)

# Put everything in one downloadable zip file
zip_path = Path("/kaggle/working/complexity_models.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in OUTPUT_DIR.iterdir():
        zipf.write(file_path, arcname=file_path.name)

print(f"Saved files in: {OUTPUT_DIR}")
print(f"Download zip: {zip_path}")

display(FileLink(str(zip_path)))